# RQ4: Limitations

**Research Question**: What are the causes of unsuccessful generalization attempts?

This notebook analyzes the limitations of test generalization by examining filtering causes and processing failures:
- **Exclusion Analysis**: Overall filtering of tests, assertions, and generalizations by variant
- **Filter Effectiveness**: Detailed breakdown of which filters cause exclusions
- **SPF Failures**: Symbolic PathFinder execution error categorization
- **Test Failures**: Runtime test execution failures by variant
- **Pipeline Analysis**: Processing stage failures in extended dataset evaluation

In [ ]:
from teralizer.config import db_config
from teralizer.rq4_limitations import (
    get_exclusions_summary_data,
    get_filtering_exclusions_data,
    get_spf_failures_data,
    get_test_failures_data,
    get_processing_failures_summary_data,
    get_processing_failure_causes_data,
    compute_exclusion_percentages,
    compute_filtering_exclusions_summary,
    compute_spf_error_categorization,
    compute_test_failures_pivot,
    compute_processing_pipeline_statistics,
    compute_failure_causes_by_stage,
    generate_exclusions_summary_table,
    generate_filtering_results_table,
    generate_spf_failures_table,
    generate_test_failures_table,
    generate_processing_failures_tables,
    generate_exclusions_summary_csv,
    generate_filtering_results_csv,
    generate_spf_failures_csv,
    generate_test_failures_csv,
    generate_processing_failures_csv,
    generate_processing_failure_causes_csv,
)
from teralizer.exports import save_latex_table, save_csv_data
from IPython.display import display

import pandas as pd

# Database connections
conn_dev = db_config.get_dev_engine()  # Main evaluation dataset
conn_test = db_config.get_test_engine()  # Extended dataset

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## Overall Exclusions Summary

Analysis of included and excluded counts by variant and level (Test/Assertion/Generalization).

In [ ]:
# Get overall exclusions data
exclusions_raw = get_exclusions_summary_data(conn_dev)
exclusions_summary = compute_exclusion_percentages(exclusions_raw)

print("Overall exclusions by variant and level:")
display(
    exclusions_summary[
        [
            "variant",
            "Type",
            "Total",
            "included_count",
            "excluded_count",
            "included_pct",
            "excluded_pct",
        ]
    ]
)

# Generate LaTeX table
exclusions_table = generate_exclusions_summary_table(exclusions_summary)
print("\n=== LaTeX Table ===")
print(exclusions_table)

# Save LaTeX table
save_latex_table(exclusions_table, "tab-exclusions-summary")

# Generate and save CSV data
exclusions_csv = generate_exclusions_summary_csv(exclusions_summary)
csv_path = save_csv_data(
    exclusions_csv,
    "exclusions-summary-data",
    "Included and excluded counts by variant and level (Test/Assertion/Generalization)",
)

print(f"\nExclusions summary data exported to: {csv_path}")
print(f"Shape: {exclusions_csv.shape}")
print("Sample data:")
display(exclusions_csv.head())

## Filtering-Based Exclusions

Analysis of filtering results for tests, assertions, and generalizations by filter type and variant.

In [ ]:
# Get filtering exclusions data from main dataset
filtering_raw_main = get_filtering_exclusions_data(conn_dev)
filtering_main = compute_filtering_exclusions_summary(filtering_raw_main.copy())

print("Main dataset filtering results:")
display(
    filtering_main[
        [
            "variant",
            "Type",
            "filter_name",
            "total",
            "accept",
            "reject",
            "accept_pct",
            "reject_pct",
        ]
    ]
)

# Generate LaTeX table for main dataset
main_filtering_table = generate_filtering_results_table(
    filtering_main,
    "tab:exclusions-filtering",
    "Filtering results for tests, assertions, and generalizations by filter and (generalization) variant.",
)

print("\n=== Main Dataset Filtering Results ===\n")
print(main_filtering_table)

# Save main dataset table
save_latex_table(main_filtering_table, "tab-exclusions-filtering")

# Export CSV data for main dataset
main_filtering_csv = generate_filtering_results_csv(filtering_main, "main")
csv_path = save_csv_data(
    main_filtering_csv,
    "exclusions-filtering-data",
    "Filtering results for tests, assertions, and generalizations by filter and variant",
)
print(f"Main filtering data exported to: {csv_path}")

In [ ]:
# Try to get extended dataset filtering results
try:
    filtering_raw_extended = get_filtering_exclusions_data(conn_test)
    filtering_extended = compute_filtering_exclusions_summary(
        filtering_raw_extended.copy()
    )

    print("Extended dataset filtering results:")
    display(
        filtering_extended[
            [
                "variant",
                "Type",
                "filter_name",
                "total",
                "accept",
                "reject",
                "accept_pct",
                "reject_pct",
            ]
        ]
    )

    # Generate LaTeX table for extended dataset
    extended_filtering_table = generate_filtering_results_table(
        filtering_extended,
        "tab:exclusions-filtering-extended",
        "Filtering results of the extended dataset for tests, assertions, and generalizations.",
    )

    print("\n=== Extended Dataset Filtering Results ===\n")
    print(extended_filtering_table)

    # Save extended dataset table
    save_latex_table(extended_filtering_table, "tab-exclusions-filtering-extended")

    # Export CSV data for extended dataset
    extended_filtering_csv = generate_filtering_results_csv(
        filtering_extended, "extended"
    )
    csv_path = save_csv_data(
        extended_filtering_csv,
        "exclusions-filtering-extended-data",
        "Extended dataset filtering results",
    )
    print(f"Extended filtering data exported to: {csv_path}")

except Exception as e:
    print(f"Extended dataset filtering not available: {e}")

## SPF Execution Failures

Analysis of Symbolic PathFinder execution failures by error type.

In [ ]:
# Get SPF failures data
spf_raw = get_spf_failures_data(conn_dev)
spf_categorized = compute_spf_error_categorization(spf_raw)

print("SPF execution failures by error type:")
display(spf_categorized)

# Generate LaTeX table
spf_table = generate_spf_failures_table(spf_categorized)
print("\n=== SPF Failures Table ===\n")
print(spf_table)

# Save LaTeX table
save_latex_table(spf_table, "tab-exclusions-spf")

# Generate and save CSV data
spf_csv = generate_spf_failures_csv(spf_categorized)
csv_path = save_csv_data(
    spf_csv, "exclusions-spf-data", "SPF execution failures by error type"
)

print(f"\nSPF errors data exported to: {csv_path}")
print(f"Shape: {spf_csv.shape}")
print("Sample data:")
display(spf_csv)

## Test Execution Failures

Analysis of test execution failures by exception type and generalization variant.

In [ ]:
# Get test failures data
test_failures_raw = get_test_failures_data(conn_dev)
test_failures_pivot, ordered_variants = compute_test_failures_pivot(test_failures_raw)

print("Test execution failures by exception type and variant:")
display(test_failures_pivot)

# Generate LaTeX table
test_failures_table = generate_test_failures_table(
    test_failures_pivot, ordered_variants
)
print("\n=== Test Failures Table ===\n")
print(test_failures_table)

# Save LaTeX table
save_latex_table(test_failures_table, "tab-exclusions-test-fails")

# Generate and save CSV data
test_failures_csv = generate_test_failures_csv(test_failures_pivot, ordered_variants)
csv_path = save_csv_data(
    test_failures_csv,
    "exclusions-test-fails-data",
    "Test execution failures by exception type and variant",
)

print(f"\nTest failures data exported to: {csv_path}")
print(f"Shape: {test_failures_csv.shape}")
print("Sample data:")
display(test_failures_csv.head())

## Processing Pipeline Failures

Analysis of processing pipeline failures in the extended dataset, showing where projects fail in the processing stages.

In [ ]:
# Get processing failures data from extended dataset
try:
    processing_summary_raw = get_processing_failures_summary_data(conn_test)
    failures_per_stage, total_projects, success_count = (
        compute_processing_pipeline_statistics(processing_summary_raw)
    )

    print(
        f"Processing pipeline statistics (Total projects: {total_projects}, Successfully processed: {success_count}):"
    )
    display(failures_per_stage[["status", "Failures", "Remaining"]])

    # Get detailed failure causes
    processing_causes_raw = get_processing_failure_causes_data(conn_test)
    stage_to_causes = compute_failure_causes_by_stage(processing_causes_raw)

    print("\nFailure causes by stage:")
    for stage, causes in stage_to_causes.items():
        print(f"{stage}: {causes}")

    # Generate LaTeX tables
    summary_table, causes_table = generate_processing_failures_tables(
        failures_per_stage, total_projects, success_count, stage_to_causes
    )

    print("\n=== Processing Failures Summary Table ===\n")
    print(summary_table)

    print("\n=== Processing Failure Causes Table ===\n")
    print(causes_table)

    # Save LaTeX tables
    save_latex_table(summary_table, "tab-processing-failures-per-stage")
    save_latex_table(causes_table, "tab-processing-failure-causes")

    # Generate and save CSV data
    processing_summary_csv = generate_processing_failures_csv(
        failures_per_stage, total_projects, success_count
    )
    csv_path = save_csv_data(
        processing_summary_csv,
        "processing-failures-summary-data",
        "Processing failures and remaining projects per stage",
    )
    print(f"\nProcessing failures summary exported to: {csv_path}")

    processing_causes_csv = generate_processing_failure_causes_csv(stage_to_causes)
    csv_path = save_csv_data(
        processing_causes_csv,
        "processing-failure-causes-data",
        "Causes of processing failures per stage",
    )
    print(f"Processing failure causes exported to: {csv_path}")

except Exception as e:
    print(f"Processing pipeline analysis not available: {e}")
    print("This requires access to the extended dataset (conn_test).")

## Summary

This notebook has analyzed the limitations of the test generalization approach across multiple dimensions:

1. **Overall Exclusions**: Shows how many tests, assertions, and generalizations are filtered out by variant
2. **Filtering Results**: Details which specific filters cause exclusions and their effectiveness
3. **SPF Failures**: Categorizes symbolic execution failures by error type
4. **Test Failures**: Analyzes runtime test execution failures by variant
5. **Processing Pipeline**: Tracks where projects fail in the complete processing pipeline

All results have been exported as both LaTeX tables and CSV data for further analysis.